In [1]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

from tools import *


In [2]:
# ---------------------------------------------------------------------------
# LLM
# ---------------------------------------------------------------------------

llm = ChatOllama(
    model="qwen3.5:4b",
    temperature=0,
    reasoning=False,
    thinking=False,
)


# ---------------------------------------------------------------------------
# Agent
# ---------------------------------------------------------------------------

SYSTEM_PROMPT = """
You are an AI Support Engineer for AcmeCloud.

You help customers with questions about their account, subscription,
orders, invoices, and support tickets.

You have access to database tools containing customer-specific information.

Important rules:

1. Use database tools when the user asks about their own account.
2. Do not invent customer information.
3. If you need customer information, identify the customer first.
4. Use the available tools instead of guessing.
5. If a tool cannot find the requested information, clearly say so.
6. Keep responses concise but useful.
"""


agent = create_agent(
    model=llm,
    tools=DATABASE_TOOLS,
    system_prompt=SYSTEM_PROMPT,
)

In [3]:
import tools
result = tools.find_customer.invoke({
    "first_name": "Maham",
    "last_name": "Malik"
})

print(result)

{'found': True, 'customer': {'id': 1, 'first_name': 'Maham', 'last_name': 'Malik', 'username': 'maham_malik_1', 'email': 'maham_malik_1@example.com', 'city': 'Lahore', 'created_at': '2024-08-07T12:47:25'}}


In [4]:
# ---------------------------------------------------------------------------
# Run the agent
# ---------------------------------------------------------------------------

response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "What is the subscription status of Maham Malik?",
                }
            ]
        }
    )

In [5]:
response['messages'][-1].content

"Maham Malik's subscription is **active**.\n\nHere are the details:\n- **Plan:** Professional\n- **Status:** Active\n- **Start Date:** May 2, 2026\n- **Renewal Date:** May 2, 2027\n- **Price:** $2,499.00"

In [6]:
refund_response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "What is the subscription refund policy?",
                }
            ]
        }
    )

/home/workspace/learning/AI_Support_Eng/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10180.35it/s]


In [7]:
refund_response['messages'][-1].content

"AcmeCloud offers a **14-day refund period** for eligible purchases. Here are the key points:\n\n*   **Standard Refund:** Customers may request a refund within 14 days of their original purchase. Annual subscription purchases are generally eligible for a full refund if requested within this period.\n*   **Billing Errors:** Billing errors (such as duplicate charges or incorrect subscription charges) may qualify for a refund even after the 14-day period has expired. Support engineers must verify the transaction before promising a refund. If the issue cannot be verified, it should be escalated to the billing team.\n*   **Third-Party Purchases:** Purchases made through third-party marketplaces are subject to the marketplace's refund policies. AcmeCloud support may provide general guidance but cannot directly refund transactions processed entirely by a third party.\n\nIf you have a specific situation or need help with your own account, please let me know!"

In [11]:
print("Collection count:", vectorstore._collection.count())

Collection count: 35


In [12]:
sub_plan_res = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "What subscription plans does AcmeCloud offer?",
                }
            ]
        }
    )

In [13]:
sub_plan_res['messages'][-1].content

'AcmeCloud offers four subscription plans:\n\n1. **Free**\n   - Up to 1 team member\n   - 5 GB storage\n   - Basic project management\n   - Standard email support\n   - Limited reporting\n   - No API access\n\n2. **Starter**\n   - Up to 5 team members\n   - 50 GB storage\n   - Basic reporting\n   - Standard integrations\n   - API access\n   - Standard support\n\n3. **Professional**\n   - Up to 20 team members\n   - 500 GB storage\n   - Advanced analytics\n   - API access\n   - Advanced reporting\n   - Priority support\n\n4. **Business**\n   - Up to 100 team members\n   - 5 TB storage\n   - Advanced analytics\n   - API access\n   - Priority support\n   - Organization-level administration\n   - Advanced security controls\n\nCustomers can upgrade their subscription at any time, and the new plan becomes available immediately.'